# 03 · Auditoría de Go y figuras del informe

**Laboratorio en Python, no sistema.** Go es la fuente de verdad; este notebook es la segunda
implementación que lo controla (patrón del TP: si discrepan, hay un bug en una de las dos, y se dice
cuál). Hace cuatro cosas:

1. **Recalcula la limpieza desde el bronze** con `aire_lab` (la misma config y la misma cascada de
   reglas que `datos/bloque.go`) y la compara con `reports/limpieza.json`: leídas, procesadas,
   descartadas por regla, marcadas, fuera de rango por parámetro, filas anchas, estaciones y
   valores vistos. El silver se compara **byte a byte** y, si difiere, celda por celda.
2. **Recalcula el gold** con la receta de `configs/kmeans.json` (tolerancia 1e-9 después del
   z-score) y la estandarización (`reports/estandarizacion.json`).
3. **Audita el K-Means de Go** (`reports/kmeans.json`, `data/gold/asignaciones.csv`): asignaciones,
   tamaños, inercia, punto fijo de Lloyd, scikit-learn con asignación húngara; y
   `reports/regimenes.json` (perfil, PM medio, etiqueta ECA, horas y estaciones) con el mismo
   cálculo que `kmeans.Perfilar`; las semillas de Go (el arranque elegido y Lloyd desde sus
   centroides iniciales); además `codo.json` y los benchmarks (que sean del gold y la receta auditados).
4. **Dibuja las figuras del informe** con los nombres que espera `docs/informe/secciones/*.qmd`:
   `perfiles.png`, `niveles.png` (PM2.5 frente a PM10 por régimen: el modelo de un vistazo),
   `regimen-hora.png`, `regimen-estacion.png`, `episodio.png` (en la estación con más horas en el
   régimen más alto), `codo.png`, `speedup.png` (y `cobertura.png`). Cada una se muestra aquí y queda
   anotada en `origen.json`, con marca de agua si los datos son sintéticos.

Estados: **OK**; **DIFERENCIA** (bug en Go o en Python: el detalle dice cuál parece); **AVISO** (no
es un bug pero hay que mirarlo, p. ej. otro mínimo local); **OMITIDO** (falta la salida de Go). Si
Go todavía no corrió, las figuras usan los regímenes de scikit-learn y lo dicen.

| Entra | Sale |
|---|---|
| bronze (`AIRE_BRONZE`, `data/bronze/*.csv` o la `entrada` de `limpieza.json`), `configs/` | `reports/auditoria.md` y `.json` |
| Go: `data/silver/oefa_ancho.csv`, `data/gold/oefa_features.csv`, `data/gold/asignaciones.csv` | figuras en `docs/informe/generado/figuras/` (local) o `reports/figuras/` en Drive (Colab) |
| Go: `reports/{limpieza,estandarizacion,kmeans,regimenes,codo,benchmark_cargador,benchmark_kmeans}.json` | |

## 0. Entorno

En Colab basta con *Entorno de ejecución → Ejecutar todo*. Esta celda monta Drive (la primera vez en
cada sesión pide autorizarlo), deja el código en la rama con que se abrió el notebook, instala lo que
falte, imprime commit, raíz de los datos, CPU y RAM, y verifica que los insumos estén en
`MyDrive/aire-distribuido/`: si falta algo o está desfasado, dice qué subir y con qué comando de
`rclone`. Ver «Iterar en Colab» en `notebooks/README.md`.

In [ ]:
# ── 0. Entorno: la misma celda en los tres notebooks ──────────────────────────────────────────
# En Colab: monta Drive, deja el repo en /content/aire-distribuido en la rama con que se abrió el
# notebook (la de la URL; si no se puede leer, RAMA) con git fetch + reset --hard, instala lo que
# falte y dice qué commit, qué datos y qué máquina se usan. En local no toca git ni Drive: usa el
# repo donde está el notebook y la raíz de AIRE_RAIZ (o el repo).
RAMA = "feature/spec-pc3"   # rama del código si no se puede leer de la URL de Colab
ACTUALIZAR_CODIGO = True    # Colab: traer lo último de la rama en cada corrida (False: dejar el código como está)
RAIZ_DRIVE = "/content/drive/MyDrive/aire-distribuido"  # capas medallón en Drive
NOTEBOOK = "03"              # qué insumos se verifican al final de la celda

import importlib, importlib.util, json, os, re, subprocess, sys, urllib.parse, urllib.request
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
RAIZ_LOCAL = os.environ.get("AIRE_RAIZ")  # otra raíz de las capas (p. ej. el sintético); en Colab, sin Drive
REPO_GIT = "https://github.com/R0SEWT/aire-distribuido.git"
PAQUETES = {"pandas": "pandas", "numpy": "numpy", "matplotlib": "matplotlib", "scikit-learn": "sklearn",
            "scipy": "scipy", "joblib": "joblib"}  # Colab los trae: solo se instala lo que falte

if EN_COLAB and not RAIZ_LOCAL:
    from google.colab import drive
    drive.mount("/content/drive")  # la primera vez en cada sesión pide autorizar Drive


def rama_de_colab():
    """La rama de la URL del notebook (colab…/github/R0SEWT/aire-distribuido/blob/<rama>/notebooks/…)."""
    for host in dict.fromkeys([os.environ.get("COLAB_JUPYTER_IP"), "172.28.0.12", "172.28.0.2", "127.0.0.1"]):
        if not host:
            continue
        try:
            puerto = os.environ.get("KMP_TARGET_PORT", "9000")
            with urllib.request.urlopen(f"http://{host}:{puerto}/api/sessions", timeout=3) as r:
                texto = urllib.parse.unquote(urllib.parse.unquote(r.read().decode()))
        except Exception:
            continue
        m = re.search(r"aire-distribuido/blob/(.+?)/notebooks/", texto)
        return m.group(1) if m else None
    return None


def git(*args, repo=None):
    r = subprocess.run(["git", *(["-C", str(repo)] if repo else []), *args], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"git {' '.join(args)}: {r.stderr.strip()}")
    return r.stdout.strip()


def traer_repo(rama, destino=Path("/content/aire-distribuido")):
    """Clona la rama (la primera vez) o la pone al día con fetch + reset --hard (las siguientes)."""
    if not (destino / ".git").exists():
        subprocess.run(["rm", "-rf", str(destino)])
        git("clone", "-q", "--depth", "1", "-b", rama, REPO_GIT, str(destino))
    elif ACTUALIZAR_CODIGO:
        git("fetch", "-q", "--depth", "1", "origin", f"+refs/heads/{rama}:refs/remotes/origin/{rama}", repo=destino)
        git("checkout", "-q", "-f", "-B", rama, f"origin/{rama}", repo=destino)
        git("reset", "-q", "--hard", f"origin/{rama}", repo=destino)
    return destino


if EN_COLAB:
    RAMA_USADA, ORIGEN_RAMA = rama_de_colab(), "de la URL de Colab"
    if not RAMA_USADA:
        RAMA_USADA, ORIGEN_RAMA = RAMA, "de RAMA: no se pudo leer la URL"
    for rama in dict.fromkeys([RAMA_USADA, "develop"]):
        try:
            REPO = traer_repo(rama)
            break
        except RuntimeError as err:
            print(f"AVISO: no pude traer la rama {rama!r} ({err}); pruebo la siguiente.")
    else:
        raise RuntimeError(f"No pude clonar {REPO_GIT}: ¿hay internet en el entorno de ejecución?")
    if rama != RAMA_USADA:
        RAMA_USADA, ORIGEN_RAMA = rama, f"{RAMA_USADA!r} no existe en GitHub"
    if git("rev-parse", "--abbrev-ref", "HEAD", repo=REPO) != RAMA_USADA:  # clon viejo sin actualizar
        RAMA_USADA, ORIGEN_RAMA = git("rev-parse", "--abbrev-ref", "HEAD", repo=REPO), "clon anterior: ACTUALIZAR_CODIGO = False"
else:
    REPO = next((c for c in (Path.cwd(), Path.cwd().parent) if (c / "notebooks" / "aire_lab.py").exists()), None)
    if REPO is None:
        raise FileNotFoundError("No encuentro notebooks/aire_lab.py: abre el notebook desde notebooks/ o desde la raíz del repo.")
    RAMA_USADA, ORIGEN_RAMA = None, ""

faltan = [pip for pip, mod in PAQUETES.items() if importlib.util.find_spec(mod) is None]
if faltan:
    print("Instalando:", faltan)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)

if str(REPO / "notebooks") not in sys.path:
    sys.path.insert(0, str(REPO / "notebooks"))
RAIZ = Path(RAIZ_LOCAL) if RAIZ_LOCAL else (Path(RAIZ_DRIVE) if EN_COLAB else REPO)
DATA, REPORTS = RAIZ / "data", RAIZ / "reports"
CONFIGS = [RAIZ / "configs", REPO / "configs"]  # la de la raíz (Drive), si existe, va primero
# Figuras del informe: en local, docs/informe/generado/figuras/; en Colab, reports/figuras/ de Drive.
FIGURAS = Path(os.environ["AIRE_FIGURAS"]) if os.environ.get("AIRE_FIGURAS") else (
    RAIZ / "reports" / "figuras" if EN_COLAB else REPO / "docs" / "informe" / "generado" / "figuras")
REPORTS.mkdir(parents=True, exist_ok=True)
FIGURAS.mkdir(parents=True, exist_ok=True)

import aire_lab as al
importlib.reload(al)
CMAPS = al.estilo_figuras()
SINTETICO = al.es_sintetico(RAIZ)  # se vuelve a mirar al leer los datos (estaciones SINT-/SIN-)
ENTORNO = al.describir_entorno(REPO, RAIZ, FIGURAS, EN_COLAB, RAMA_USADA, ORIGEN_RAMA)
INSUMOS = al.verificar_insumos(NOTEBOOK, RAIZ, CONFIGS, estricto=EN_COLAB)

In [ ]:
import json, math, time, datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from IPython.display import display, Markdown
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 180)

# Tolerancias de la auditoría.
TOL_GOLD = 1e-9         # absoluta, después del z-score (spec de calibración §4)
TOL_ESTAND = 1e-9       # relativa, medias y desviaciones
TOL_PERFIL = 1e-9       # relativa, perfiles de regimenes.json
TOL_SKLEARN = 1e-3      # centroides de Go frente a los de scikit-learn, tras el emparejamiento


def buscar(nombre, *dirs):
    for d in dirs:
        if (Path(d) / nombre).exists():
            return Path(d) / nombre
    return Path(dirs[0]) / nombre


GENERADO = REPO / "docs" / "informe" / "generado"  # copia de los JSON de Go que usa el informe
GO = {
    "silver": DATA / "silver" / "oefa_ancho.csv",
    "gold": DATA / "gold" / "oefa_features.csv",
    "asignaciones": DATA / "gold" / "asignaciones.csv",
    "limpieza": REPORTS / "limpieza.json",
    "estandarizacion": REPORTS / "estandarizacion.json",
    "kmeans": REPORTS / "kmeans.json",
    "regimenes": REPORTS / "regimenes.json",
    "centroides_iniciales": REPORTS / "centroides_iniciales.json",
    "codo": buscar("codo.json", REPORTS, GENERADO),
    "benchmark_cargador": buscar("benchmark_cargador.json", REPORTS, GENERADO),
    "benchmark_kmeans": buscar("benchmark_kmeans.json", REPORTS, GENERADO),
}
display(pd.DataFrame({"ruta": {k: str(v) for k, v in GO.items()}, "existe": {k: v.exists() for k, v in GO.items()}}))

CHEQUEOS = []


def chequeo(nombre, ok, detalle, omitido=False, aviso=False):
    estado = "OMITIDO" if omitido else ("OK" if ok else ("AVISO" if aviso else "DIFERENCIA"))
    CHEQUEOS.append({"chequeo": nombre, "estado": estado, "detalle": detalle})
    print(f"[{estado}] {nombre}: {detalle}")


def leer_json(ruta):
    with open(ruta, encoding="utf-8") as fh:
        return json.load(fh)


def rel(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return float(np.max(np.abs(a - b) / np.maximum(1.0, np.abs(a)))) if a.size else 0.0


def guardar(fig, nombre, fuente):
    al.marca_sintetico(fig, SINTETICO)
    fig.savefig(FIGURAS / nombre, dpi=150)
    al.registrar_figura(FIGURAS, nombre, SINTETICO, fuente, "03_auditoria_figuras")
    plt.show()

## 1. Receta y bronze

In [ ]:
try:
    receta, origen_receta = al.leer_receta([c / "kmeans.json" for c in CONFIGS])
except FileNotFoundError:
    receta, origen_receta = leer_json(REPORTS / "kmeans_propuesto.json"), str(REPORTS / "kmeans_propuesto.json")
    print("AVISO: no hay configs/kmeans.json; se usa la propuesta del notebook 02.")
FEATS, LOGS, NIVEL, K = receta["features"], receta.get("log1p", []), receta.get("nivel", "absoluto"), int(receta["k"])
print(f"Receta ({origen_receta}):", json.dumps(receta))

lim_go = leer_json(GO["limpieza"]) if GO["limpieza"].exists() else None
BRONZE = os.environ.get("AIRE_BRONZE")
if BRONZE is None:
    bronces = sorted((DATA / "bronze").glob("*.csv"), key=lambda p: p.stat().st_size, reverse=True)
    entrada = Path(lim_go["entrada"]) if lim_go and lim_go.get("entrada") else None
    candidatos = bronces + ([entrada] if entrada and entrada.is_absolute() else
                            [REPO / entrada, RAIZ / entrada] if entrada else [])
    BRONZE = next((c for c in candidatos if Path(c).exists()), None)
BRONZE = Path(BRONZE) if BRONZE else None
print("Bronze:", BRONZE)
SINTETICO = SINTETICO or (BRONZE is not None and al.es_sintetico(bronze=BRONZE))

## 2. Limpieza: el prototipo frente a `reports/limpieza.json`

La misma config (`configs/limpieza.json`) y la misma cascada que `datos/bloque.go`, con los mismos
nombres de regla. Se comparan todos los conteos del reporte de Go.

In [ ]:
cfg, origen_cfg = al.cargar_config([c / "limpieza.json" for c in CONFIGS])
print("Config de limpieza:", origen_cfg)
largo = ancho_py = rep_py = None
if BRONZE is not None:
    t0 = time.perf_counter()
    largo, info_lectura = al.leer_bronze(BRONZE, cfg)
    validos, rep_py = al.limpiar(largo, cfg, info_lectura)
    ancho_py = al.pivotar(validos, rep_py)
    print(f"{BRONZE.name}: {rep_py['filas_largas']['leidas']:,} filas largas → {len(ancho_py):,} filas anchas "
          f"en {time.perf_counter() - t0:.1f} s")
    SINTETICO = SINTETICO or al.es_sintetico(estaciones=ancho_py["estacion"].unique())
else:
    print("AVISO: no hay bronze; no se puede recalcular la limpieza.")

if rep_py is not None and lim_go is not None:
    chequeo("cabecera, BOM y delimitador",
            lim_go.get("cabecera") == rep_py["cabecera"] and lim_go.get("bom") == rep_py["bom"]
            and lim_go.get("delimitador") == rep_py["delimitador"],
            f"go BOM={lim_go.get('bom')} delim={lim_go.get('delimitador')!r} · python BOM={rep_py['bom']} delim={rep_py['delimitador']!r}")
    fl_py, fl_go = rep_py["filas_largas"], lim_go["filas_largas"]
    chequeo("filas largas (leídas, procesadas, descartadas)",
            all(fl_py[k] == fl_go.get(k) for k in ["leidas", "procesadas", "descartadas"]),
            " · ".join(f"{k} py {fl_py[k]:,} go {fl_go.get(k):,}" for k in ["leidas", "procesadas", "descartadas"]))
    dpr_go = fl_go.get("descartadas_por_regla", {})
    reglas = list(dict.fromkeys(al.NOMBRES_REGLAS + list(dpr_go)))
    t = pd.DataFrame({"python": [fl_py["descartadas_por_regla"].get(r) for r in reglas],
                      "go": [dpr_go.get(r) for r in reglas]}, index=reglas)
    # Una regla que Go todavía no tiene y que en Python no descartó nada no es una diferencia.
    t["igual"] = (t["python"] == t["go"]) | (t["go"].isna() & (t["python"] == 0))
    display(t)
    solo = [r for r in reglas if r not in dpr_go or r not in fl_py["descartadas_por_regla"]]
    chequeo("descartadas por regla", bool(t["igual"].all()),
            f"las {int(t['go'].notna().sum())} reglas de Go iguales" + (f" (solo en Python, sin casos: {solo})" if solo else "")
            if t["igual"].all() else f"distintas en {list(t.index[~t['igual']])}; en un solo lado: {solo}")
    mk_go = fl_go.get("marcadas_por_regla") or {}
    mk_dif = {k: (fl_py["marcadas_por_regla"].get(k), v) for k, v in mk_go.items() if fl_py["marcadas_por_regla"].get(k) != v}
    extra = {k: v for k, v in fl_py["marcadas_por_regla"].items() if k not in mk_go}
    chequeo("marcadas por regla", not mk_dif,
            f"go {mk_go}" + (f"; diferencias (py, go): {mk_dif}" if mk_dif else "") + (f"; Python marca además {extra} (Go no las escribe)" if extra else ""))
    chequeo("fuera de rango por parámetro", fl_py["fuera_de_rango_por_parametro"] == fl_go.get("fuera_de_rango_por_parametro"),
            f"python {fl_py['fuera_de_rango_por_parametro']} · go {fl_go.get('fuera_de_rango_por_parametro')}")
    fa_py, fa_go = rep_py["filas_anchas"], lim_go.get("filas_anchas", {})
    chequeo("filas anchas (total, PM2.5 > PM10, hora inferida)",
            fa_py["total"] == fa_go.get("total") and fa_py["pm2_5_mayor_pm10"] == fa_go.get("pm2_5_mayor_pm10")
            and fa_py.get("hora_inferida") == fa_go.get("hora_inferida"),
            f"python total {fa_py['total']:,}, pm2_5>pm10 {fa_py['pm2_5_mayor_pm10']} · go total {fa_go.get('total')}, "
            f"pm2_5>pm10 {fa_go.get('pm2_5_mayor_pm10')} · hora inferida py {fa_py.get('hora_inferida')} go {fa_go.get('hora_inferida')}")
    chequeo("parámetros presentes", rep_py["parametros_presentes"] == lim_go.get("parametros_presentes"),
            f"{lim_go.get('parametros_presentes')}")
    dif_est = []
    for e in sorted(set(lim_go.get("estaciones", {})) | set(rep_py["estaciones"])):
        a, b = rep_py["estaciones"].get(e, {}), lim_go.get("estaciones", {}).get(e, {})
        for k in ["largas_leidas", "largas_procesadas", "anchas", "desde", "hasta", "ubicacion"]:
            if a.get(k) != b.get(k):
                dif_est.append(f"{e}.{k}: py {a.get(k)} go {b.get(k)}")
    chequeo("estaciones (leídas, procesadas, anchas, fechas, ubicación)", not dif_est,
            f"{len(lim_go.get('estaciones', {}))} estaciones iguales" if not dif_est else "; ".join(dif_est[:8]))
    chequeo("valores vistos (parámetros, periodos, unidades)", rep_py["vistos"] == lim_go.get("vistos"),
            "iguales" if rep_py["vistos"] == lim_go.get("vistos") else
            "; ".join(f"{k}: py {rep_py['vistos'][k]} go {lim_go.get('vistos', {}).get(k)}" for k in rep_py["vistos"]
                      if rep_py["vistos"][k] != lim_go.get("vistos", {}).get(k)))
else:
    chequeo("limpieza", False, "falta el bronze o reports/limpieza.json de Go", omitido=True)

## 3. Silver: byte a byte y, si difiere, celda por celda

In [ ]:
silver_go = al.leer_csv_contrato(GO["silver"]) if GO["silver"].exists() else None
if ancho_py is not None and silver_go is not None:
    ruta_py = Path(__import__("tempfile").gettempdir()) / "auditoria_silver_python.csv"  # no en Drive
    al.escribir_csv_contrato(ancho_py, ruta_py)
    sha_py, sha_go = al.sha256(ruta_py), al.sha256(GO["silver"])
    ruta_py.unlink()
    if sha_py == sha_go:
        chequeo("silver byte a byte", True, f"idéntico (sha256 {sha_go[:16]}…)")
    else:
        m = ancho_py.merge(silver_go, on=["estacion", "fecha_hora"], how="outer", indicator=True, suffixes=("_py", "_go"))
        solo_py, solo_go = m[m["_merge"] == "left_only"], m[m["_merge"] == "right_only"]
        ambos = m[m["_merge"] == "both"]
        filas_dif = []
        for c in [c for c in ancho_py.columns[2:] if c in silver_go.columns]:
            a, b = ambos[c + "_py"].to_numpy(float), ambos[c + "_go"].to_numpy(float)
            malas = (np.isnan(a) != np.isnan(b)) | (~np.isnan(a) & ~np.isnan(b) & (a != b))
            if malas.any():
                filas_dif.append(ambos.loc[malas, ["estacion", "fecha_hora", c + "_py", c + "_go"]].head(10))
        cab_igual = list(silver_go.columns) == list(ancho_py.columns)
        chequeo("silver byte a byte", False,
                f"cabecera {'igual' if cab_igual else 'DISTINTA'}; {len(solo_py):,} claves solo en Python, {len(solo_go):,} solo "
                f"en Go; {len(filas_dif)} columnas con valores distintos (abajo). Si los valores son iguales, la "
                "diferencia es de formato (FormatFloat 'g' -1 o el orden de las filas)")
        for d in filas_dif:
            display(d)
        if len(solo_py) or len(solo_go):
            display(pd.concat([solo_py.head(5), solo_go.head(5)])[["estacion", "fecha_hora", "_merge"]])
else:
    chequeo("silver", False, "falta el bronze o el silver de Go", omitido=True)

## 4. Gold y estandarización

In [ ]:
base_silver = ancho_py if ancho_py is not None else silver_go
gold_py, stats_py = al.construir_gold(base_silver, receta) if base_silver is not None else (None, None)
gold_go = al.leer_csv_contrato(GO["gold"]) if GO["gold"].exists() else None
if gold_py is not None and gold_go is not None:
    chequeo("cabecera del gold", list(gold_go.columns) == ["estacion", "fecha_hora"] + FEATS, f"go {list(gold_go.columns)}")
    m = gold_py.merge(gold_go, on=["estacion", "fecha_hora"], how="outer", indicator=True, suffixes=("_py", "_go"))
    n_solo = int((m["_merge"] != "both").sum())
    chequeo("filas del gold (regla 8)", n_solo == 0, f"python {len(gold_py):,} · go {len(gold_go):,} · {n_solo} claves en un solo lado")
    ambos = m[m["_merge"] == "both"]
    D = np.abs(ambos[[f + "_py" for f in FEATS]].to_numpy(float) - ambos[[f + "_go" for f in FEATS]].to_numpy(float))
    peor = float(np.nanmax(D)) if D.size else float("nan")
    malas = (D > TOL_GOLD).any(axis=1)
    chequeo("valores del gold", not malas.any(), f"diferencia máxima {peor:.2e} (tolerancia {TOL_GOLD:g}); {int(malas.sum())} filas fuera"
            + (" (las filas no coinciden: cambian la media y la desviación; resolver primero las claves)" if n_solo else ""))
    if malas.any():
        display(ambos.loc[malas].head(20))
    if silver_go is not None:  # el paso silver -> gold aislado: la receta sobre el silver de Go
        g2, _ = al.construir_gold(silver_go, receta)
        m2 = g2.merge(gold_go, on=["estacion", "fecha_hora"], how="outer", indicator=True, suffixes=("_py", "_go"))
        a2 = m2[m2["_merge"] == "both"]
        d2 = float(np.nanmax(np.abs(a2[[f + "_py" for f in FEATS]].to_numpy(float) - a2[[f + "_go" for f in FEATS]].to_numpy(float)))) if len(a2) else float("nan")
        chequeo("gold desde el silver de Go", int((m2["_merge"] != "both").sum()) == 0 and d2 <= TOL_GOLD,
                f"{int((m2['_merge'] != 'both').sum())} claves en un solo lado; diferencia máxima {d2:.2e}")
    if lim_go is not None:
        fa = lim_go.get("filas_anchas", {})
        por_est_py = gold_py["estacion"].value_counts()
        dif = {e: (int(por_est_py.get(e, 0)), r.get("completas_gold")) for e, r in lim_go.get("estaciones", {}).items()
               if int(por_est_py.get(e, 0)) != r.get("completas_gold")}
        chequeo("completas para el gold (total y por estación)", fa.get("completas_gold") == len(gold_py) and not dif,
                f"python {len(gold_py):,} · go {fa.get('completas_gold')}" + (f"; por estación (py, go): {dif}" if dif else ""))
else:
    chequeo("gold", False, "falta el silver o el gold de Go", omitido=True)

est_go = leer_json(GO["estandarizacion"]) if GO["estandarizacion"].exists() else None
if stats_py is not None and est_go is not None:
    chequeo("receta en estandarizacion.json",
            est_go.get("features") == FEATS and (est_go.get("log1p") or []) == LOGS and est_go.get("nivel") == NIVEL
            and est_go.get("ddof") == 0,
            f"go features={est_go.get('features')} log1p={est_go.get('log1p')} nivel={est_go.get('nivel')} ddof={est_go.get('ddof')}")
    chequeo("n de la estandarización", est_go.get("n") == stats_py["n"], f"python {stats_py['n']:,} · go {est_go.get('n')}")
    r_mu, r_sd = rel(stats_py["media"], est_go["media"]), rel(stats_py["desv"], est_go["desv"])
    if NIVEL == "estacion":
        for e, s in (est_go.get("por_estacion") or {}).items():
            r_mu = max(r_mu, rel(stats_py["por_estacion"][e]["media"], s["media"]))
            r_sd = max(r_sd, rel(stats_py["por_estacion"][e]["desv"], s["desv"]))
    chequeo("medias de la estandarización", r_mu <= TOL_ESTAND, f"diferencia relativa máxima {r_mu:.2e}")
    n = stats_py["n"]
    pista = (" (Go usaría n−1; el contrato es poblacional)" if r_sd > TOL_ESTAND and n > 1 and
             np.allclose(np.asarray(est_go["desv"]) / np.asarray(stats_py["desv"]), math.sqrt(n / (n - 1)), rtol=1e-6) else "")
    chequeo("desviaciones de la estandarización", r_sd <= TOL_ESTAND, f"diferencia relativa máxima {r_sd:.2e}{pista}")
else:
    chequeo("estandarización", False, "falta reports/estandarizacion.json de Go", omitido=True)

## 5. K-Means de Go

- **Trazabilidad**: el sha256 del gold en `kmeans.json`, `regimenes.json` y `codo.json` es el del
  gold auditado, y la receta de `kmeans.json` es la de `configs/kmeans.json`.
- **Asignaciones**: cada fila del gold al centroide más cercano (euclídea al cuadrado, empate al
  menor índice) frente a `asignaciones.csv`; Go las calcula con los centroides finales.
- **Tamaños, inercia** (con los centroides finales) e **inercia por iteración** que no crece.
- **Punto fijo**: si Lloyd paró por tolerancia, cada centroide es la media de sus filas salvo el
  último movimiento; scikit-learn arrancando de los centroides de Go casi no los mueve.
- **scikit-learn independiente** (k-means++, 10 arranques, hasta converger: `tol = 0`): mismos centroides tras emparejar etiquetas
  con la asignación húngara. Si difieren con igual inercia, es otro mínimo local, no un bug.
- **Semillas de Go**: el arranque elegido es el de menor inercia (si empatan bit a bit, la semilla
  menor), y Lloyd de scikit-learn desde los mismos centroides iniciales
  (`reports/centroides_iniciales.json`) llega, semilla por semilla, a la inercia que reporta Go.

In [ ]:
km_go = leer_json(GO["kmeans"]) if GO["kmeans"].exists() else None
asig_go = al.leer_csv_contrato(GO["asignaciones"]) if GO["asignaciones"].exists() else None
gold_ref = gold_go if gold_go is not None else gold_py
Cz_go, lab_go = None, None
if km_go is not None and gold_ref is not None:
    sha_gold = al.sha256(GO["gold"]) if GO["gold"].exists() else None
    rg_go = leer_json(GO["regimenes"]) if GO["regimenes"].exists() else {}
    co_go = leer_json(GO["codo"]) if GO["codo"].exists() else {}
    shas = {n: (j.get("gold") or {}).get("sha256") for n, j in [("kmeans", km_go), ("regimenes", rg_go), ("codo", co_go)] if j}
    chequeo("el modelo se entrenó con este gold", all(s == sha_gold for s in shas.values()),
            f"gold {str(sha_gold)[:12]}… · " + ", ".join(f"{n} {str(s)[:12]}…" for n, s in shas.items()))
    rec_km = km_go.get("receta", {})
    claves_receta = ["features", "log1p", "nivel", "k", "semilla", "tolerancia", "max_iter", "chunk"] + (["semillas"] if "semillas" in receta else [])
    chequeo("receta de kmeans.json = configs/kmeans.json", all(rec_km.get(c) == receta.get(c) for c in claves_receta),
            "; ".join(f"{c}: go {rec_km.get(c)} config {receta.get(c)}" for c in claves_receta if rec_km.get(c) != receta.get(c)) or "iguales")
    Cz_go = np.array(km_go["centroides_estandarizados"], dtype=float)
    X = gold_ref[FEATS].to_numpy(float)
    k_ef = int(km_go.get("k_efectivo", K))
    chequeo("forma de los centroides", Cz_go.shape == (k_ef, len(FEATS)), f"go {Cz_go.shape} · k efectivo {k_ef}, D {len(FEATS)}")
    lab_py, d2 = al.asignar(X, Cz_go)
    if asig_go is not None:
        j = gold_ref[["estacion", "fecha_hora"]].assign(_lab=lab_py).merge(asig_go, on=["estacion", "fecha_hora"], how="outer", indicator=True)
        n_solo = int((j["_merge"] != "both").sum())
        dif = j[(j["_merge"] == "both") & (j["_lab"] != j["regimen"])]
        chequeo("asignaciones", n_solo == 0 and len(dif) == 0,
                f"{len(dif):,} filas con otro régimen · {n_solo} claves en un solo lado de {len(asig_go):,}")
        if len(dif):
            display(dif.head(20))
        lab_go = j.loc[j["_merge"] == "both", "regimen"].to_numpy(int) if n_solo == 0 else lab_py
    else:
        chequeo("asignaciones", False, "falta data/gold/asignaciones.csv", omitido=True)
        lab_go = lab_py
    chequeo("tamaños", list(np.bincount(lab_py, minlength=k_ef)) == list(km_go.get("tamanos", [])),
            f"python {np.bincount(lab_py, minlength=k_ef).tolist()} · go {km_go.get('tamanos')}")
    r_in = abs(d2.sum() - km_go["inercia"]) / max(1e-12, abs(km_go["inercia"]))
    chequeo("inercia con los centroides finales", r_in <= 1e-9, f"python {d2.sum():.9f} · go {km_go['inercia']:.9f} (rel {r_in:.1e})")
    its = np.asarray(km_go.get("inercias_por_iteracion", []), float)
    crece = int((np.diff(its) > 1e-9 * np.maximum(1, its[:-1])).sum()) if len(its) > 1 else 0
    chequeo("la inercia no crece entre iteraciones", crece == 0, f"{len(its)} iteraciones, paro «{km_go.get('paro')}», {crece} subidas")
    umbral = km_go.get("tol_abs", 1e-6) + km_go.get("tol_rel", 0) * float(np.abs(Cz_go).max())
    medias = np.array([X[lab_go == q].mean(axis=0) if (lab_go == q).any() else Cz_go[q] for q in range(k_ef)])
    pf = float(np.abs(medias - Cz_go).max())
    chequeo("punto fijo de Lloyd", pf <= max(100 * umbral, 1e-9) or km_go.get("paro") != "tolerancia",
            f"máx |media de las filas − centroide| = {pf:.2e} (umbral de paro de Go {umbral:.1e}, paro «{km_go.get('paro')}»)",
            aviso=km_go.get("paro") != "tolerancia")
    km_fijo = KMeans(n_clusters=k_ef, init=Cz_go, n_init=1, max_iter=300, tol=0).fit(X)
    mov = float(np.abs(km_fijo.cluster_centers_ - Cz_go).max())
    chequeo("scikit-learn desde los centroides de Go", mov <= max(100 * umbral, 1e-9), f"se mueven {mov:.2e} en {km_fijo.n_iter_} iteraciones",
            aviso=True)
    # tol = 0: hasta que las etiquetas no cambian. Con su tol por defecto (1e-4 de la varianza) scikit-learn para
    # antes que Go (1e-6) y la comparación mezclaría la tolerancia con la implementación.
    km_sk = KMeans(n_clusters=k_ef, init="k-means++", n_init=10, tol=0, max_iter=1000,
                   random_state=int(receta.get("semilla", 42))).fit(X)
    perm = al.emparejar(Cz_go, km_sk.cluster_centers_)
    dmax = float(np.abs(km_sk.cluster_centers_[perm] - Cz_go).max())
    ari = adjusted_rand_score(lab_go, km_sk.labels_)
    rel_in = (km_sk.inertia_ - d2.sum()) / d2.sum()
    det = f"máx |Δ| tras emparejar {dmax:.2e} (tolerancia {TOL_SKLEARN:g}); ARI {ari:.4f}; inercia sklearn/go − 1 = {rel_in:+.2%}"
    in_sk = float(al.asignar(X, km_sk.cluster_centers_)[1].sum())
    sem_sk = [a["semilla"] for a in km_go.get("arranques") or [] if abs(a["inercia"] - in_sk) <= 1e-9 * abs(in_sk)]
    if dmax > TOL_SKLEARN and sem_sk:
        det += f"; la inercia de scikit-learn ({in_sk:.6f}) es la de Go con las semillas {sem_sk}: Go también visita ese mínimo y elige el menor"
    if dmax > TOL_SKLEARN:
        det += (". Otro mínimo local de igual calidad: no es un bug" if abs(rel_in) < 1e-3 else
                ". scikit-learn (10 arranques) encuentra menor inercia: Go quedó en un mínimo local peor con su único "
                "k-means++. Lloyd de Go está bien (punto fijo); es la inicialización" if rel_in < 0 else
                ". Go encuentra menor inercia que scikit-learn")
    chequeo("centroides frente a scikit-learn", dmax <= TOL_SKLEARN, det, aviso=dmax > TOL_SKLEARN and pf <= max(100 * umbral, 1e-9))
    if est_go is not None and NIVEL == "absoluto" and km_go.get("centroides_des_estandarizados") is not None:
        r_ce = rel(al.desestandarizar(Cz_go, est_go), km_go["centroides_des_estandarizados"])
        chequeo("centroides des-estandarizados", r_ce <= 1e-9, f"diferencia relativa máxima {r_ce:.2e}")
    # Semillas: el arranque elegido es el de menor inercia (empate bit a bit: la semilla menor), y Lloyd en
    # scikit-learn desde los mismos centroides iniciales llega a la misma inercia, semilla por semilla.
    arr = km_go.get("arranques") or []
    if arr:
        el = arr[int(km_go.get("arranque_elegido", 0))]
        menor = min(a["inercia"] for a in arr)
        empate = sorted(a["semilla"] for a in arr if a["inercia"] == menor)
        chequeo("arranque elegido entre las semillas",
                el["inercia"] == menor and el["semilla"] == empate[0] == km_go.get("semilla") and el["inercia"] == km_go["inercia"]
                and len(arr) == int(receta.get("semillas", len(arr))),
                f"{len(arr)} arranques (semillas {min(a['semilla'] for a in arr)}–{max(a['semilla'] for a in arr)}); elegida la "
                f"{el['semilla']}, inercia {el['inercia']:.6f} = la menor; {len(empate)} semillas empatan bit a bit en ella; "
                f"inercias distintas: {sorted({round(a['inercia'], 6) for a in arr})}")
        ini_go = leer_json(GO["centroides_iniciales"]) if GO["centroides_iniciales"].exists() else None
        if ini_go is not None:
            por_sem = {i["semilla"]: i for i in ini_go.get("inicios", [])}
            peor_rel, d_el, n_rep, falta = 0.0, float("nan"), 0, []
            for a in arr:
                i0 = por_sem.get(a["semilla"])
                if i0 is None or i0.get("sha256") != a.get("centroides_iniciales_sha256"):
                    falta.append(a["semilla"])
                    continue
                C0 = np.array(i0["centroides"], dtype=float)
                km0 = KMeans(n_clusters=len(C0), init=C0, n_init=1, algorithm="lloyd", tol=0,
                             max_iter=10 * int(receta.get("max_iter", 100))).fit(X)
                r0 = abs(al.asignar(X, km0.cluster_centers_)[1].sum() - a["inercia"]) / abs(a["inercia"])
                peor_rel = max(peor_rel, r0)
                n_rep += int(r0 <= 1e-6)
                if a is el:
                    d_el = float(np.abs(km0.cluster_centers_ - Cz_go).max())
            mismo_gold = ini_go.get("gold_sha256") == sha_gold
            chequeo("Lloyd desde los centroides iniciales de Go", n_rep == len(arr) and not falta and mismo_gold,
                    f"{n_rep} de {len(arr)} arranques reproducen la inercia de Go (diferencia relativa máxima {peor_rel:.1e}); "
                    f"semilla elegida: máx |Δ centroide| {d_el:.1e}"
                    + (f"; sin centroides iniciales o con otro sha256: {falta}" if falta else "")
                    + ("" if mismo_gold else "; centroides_iniciales.json es de otro gold"))
        else:
            chequeo("Lloyd desde los centroides iniciales de Go", False, "falta reports/centroides_iniciales.json", omitido=True)
else:
    chequeo("K-Means de Go", False, "falta reports/kmeans.json de Go", omitido=True)

## 6. `regimenes.json`: el perfil de `kmeans.Perfilar`

In [ ]:
if GO["regimenes"].exists() and lab_go is not None and gold_go is not None and est_go is not None:
    rg_go = leer_json(GO["regimenes"])
    regs_py = al.perfilar(gold_go, lab_go, len(Cz_go), est_go)
    malos = []
    for a, b in zip(regs_py, rg_go.get("regimenes", [])):
        r = b.get("regimen")
        for c in ["tamano", "eca", "por_hora", "por_estacion"]:
            if a.get(c) != b.get(c):
                malos.append(f"R{r} {c}: py {a.get(c)} go {b.get(c)}")
        if (a.get("eca_por") or []) != (b.get("eca_por") or []):
            malos.append(f"R{r} eca_por")
        for c in ["fraccion", "pm2_5_medio", "pm10_medio"]:
            if (a.get(c) is None) != (b.get(c) is None) or (a.get(c) is not None and rel([a[c]], [b[c]]) > TOL_PERFIL):
                malos.append(f"R{r} {c}: py {a.get(c)} go {b.get(c)}")
        for f in FEATS:
            if rel([a["perfil"][f]], [b["perfil"][f]]) > TOL_PERFIL:
                malos.append(f"R{r} perfil.{f}: py {a['perfil'][f]} go {b['perfil'][f]}")
    if len(regs_py) != len(rg_go.get("regimenes", [])):
        malos.append(f"python {len(regs_py)} regímenes, go {len(rg_go.get('regimenes', []))}")
    if rg_go.get("features") != FEATS or rg_go.get("nivel") != NIVEL:
        malos.append(f"features/nivel: go {rg_go.get('features')} {rg_go.get('nivel')}")
    # por_hora es la hora de fecha_hora: con hora por posición, la convención de configs/limpieza.json.
    conv_cfg = (cfg.get("hora_por_posicion") or {}).get("convencion")
    if rg_go.get("convencion_hora") and conv_cfg and rg_go["convencion_hora"] != conv_cfg:
        malos.append(f"convencion_hora: go {rg_go['convencion_hora']} · configs/limpieza.json {conv_cfg}")
    chequeo("regimenes.json (perfil, PM medio, ECA, horas, estaciones)", not malos,
            f"iguales; convención de la hora «{rg_go.get('convencion_hora', '—')}»" if not malos else "; ".join(malos[:8]))
    display(pd.DataFrame([{k: r.get(k) for k in ["regimen", "tamano", "pm2_5_medio", "pm10_medio", "eca"]} for r in rg_go["regimenes"]]))
else:
    chequeo("regimenes.json", False, "falta regimenes.json, asignaciones, gold o estandarización", omitido=True)

## 7. `codo.json` y benchmarks

In [ ]:
codo_go = leer_json(GO["codo"]) if GO["codo"].exists() else None
if codo_go is not None and gold_ref is not None:
    puntos = codo_go.get("puntos", [])
    Xc = gold_ref[FEATS].to_numpy(float)
    paso_c = max(1, math.ceil(len(Xc) / max(1, puntos[0]["n"]))) if puntos else 1
    filas = []
    for p in puntos:
        sk = KMeans(n_clusters=p["k"], init="k-means++", n_init=10, random_state=int(receta.get("semilla", 42))).fit(Xc[::paso_c])
        filas.append({"k": p["k"], "inercia_go": p["inercia"], "inercia_sklearn_10_arranques": sk.inertia_,
                      "go/sklearn": p["inercia"] / sk.inertia_, "paro": p.get("paro")})
    tc = pd.DataFrame(filas)
    display(tc.round(4))
    peor = float(tc["go/sklearn"].max()) if len(tc) else float("nan")
    chequeo("codo.json frente a scikit-learn", peor <= 1.02,
            f"inercia de Go / mejor de scikit-learn: máximo {peor:.3f} en k = {int(tc.loc[tc['go/sklearn'].idxmax(), 'k']) if len(tc) else '—'}"
            + ("" if peor <= 1.02 else "; con un solo k-means++ Go puede quedar en un mínimo local: el codo se ve más ruidoso"),
            aviso=True)
else:
    chequeo("codo.json", False, "falta reports/codo.json", omitido=True)

SHA_GOLD = al.sha256(GO["gold"]) if GO["gold"].exists() else None
SHA_RECETA = al.sha256(origen_receta) if Path(origen_receta).exists() else None
NOTA_BENCH_CARGADOR = ""
bc = leer_json(GO["benchmark_cargador"]) if GO["benchmark_cargador"].exists() else None
if bc is not None:
    shas = {m.get("sha256_silver") for m in bc.get("mediciones", [])}
    det = f"identicos={bc.get('identicos')}; {len(bc.get('mediciones', []))} valores de W"
    mismo_silver = GO["silver"].exists() and shas == {al.sha256(GO["silver"])}
    if GO["silver"].exists() and bc.get("entrada") == (lim_go or {}).get("entrada"):
        det += "; sha del silver " + ("= el auditado" if mismo_silver else "≠ el auditado")
    chequeo("cargador: mismo silver y gold para todo W", bool(bc.get("identicos")), det)
    # El cargador escribe también el gold: si la receta cambió después del benchmark, su gold es otro.
    mismo_gold = {m.get("sha256_gold") for m in bc.get("mediciones", [])} == {SHA_GOLD}
    misma_receta = ((bc.get("configs") or {}).get("kmeans") or {}).get("sha256") == SHA_RECETA
    if not (mismo_gold and misma_receta):
        NOTA_BENCH_CARGADOR = (f" El benchmark del cargador es del {str(bc.get('fecha', '?'))[:10]}, con la receta anterior: "
                               "mismo silver, otro gold.")
    chequeo("cargador: el benchmark es de la receta auditada", mismo_gold and misma_receta,
            "gold y configs/kmeans.json del benchmark = los auditados" if mismo_gold and misma_receta else
            f"el benchmark del cargador ({bc.get('fecha', '?')}) corrió con otra receta: gold {'=' if mismo_gold else '≠'} el "
            f"auditado, configs/kmeans.json {'=' if misma_receta else '≠'} el actual; el silver "
            f"{'es el mismo' if mismo_silver else 'tampoco coincide'}. No es un bug: sus tiempos son de esa corrida "
            "(bronze → silver → gold con la receta de entonces); conviene repetirlo con la receta final antes de citarlo",
            aviso=True)
else:
    chequeo("benchmark_cargador.json", False, "no está", omitido=True)
bk = leer_json(GO["benchmark_kmeans"]) if GO["benchmark_kmeans"].exists() else None
if bk is not None:
    dets = bk.get("determinismo", [])
    chequeo("K-Means: idéntico bit a bit para todo W", bool(dets) and all(d.get("identico_bit_a_bit_para_todo_w") for d in dets),
            "; ".join(f"{d.get('experimento')} chunk {d.get('chunk')}: {d.get('identico_bit_a_bit_para_todo_w')}" for d in dets))
    # Que el benchmark sea del gold y la receta auditados: con el mismo chunk y la misma semilla, su corrida
    # hasta converger tiene que dar la inercia de kmeans.json.
    rec_bk = bk.get("receta") or {}
    dif_rec = [c for c in ["features", "log1p", "nivel", "k", "semilla", "tolerancia", "max_iter"] if rec_bk.get(c) != receta.get(c)]
    conv = [d for d in dets if d.get("experimento") == "convergencia" and d.get("chunk") == receta.get("chunk")]
    r_conv = (abs(conv[0]["inercia"] - km_go["inercia"]) / abs(km_go["inercia"])) if conv and km_go else float("nan")
    chequeo("K-Means: el benchmark es del gold y la receta auditados",
            (bk.get("datos") or {}).get("sha256") == SHA_GOLD and not dif_rec and (not conv or r_conv <= 1e-12),
            f"{GO['benchmark_kmeans'].parent.name}/{GO['benchmark_kmeans'].name}: gold "
            f"{'=' if (bk.get('datos') or {}).get('sha256') == SHA_GOLD else '≠'} el auditado; receta "
            + (f"distinta en {dif_rec}" if dif_rec else "igual")
            + (f"; inercia de «convergencia» con chunk {receta.get('chunk')} frente a kmeans.json: diferencia relativa {r_conv:.1e}"
               if conv and km_go else "")
            + ("" if (bk.get("datos") or {}).get("sha256") == SHA_GOLD and not dif_rec else
               ". No es un bug: el benchmark es de otra corrida; repetir cmd/benchmark sobre este gold antes de citarlo"),
            aviso=not ((bk.get("datos") or {}).get("sha256") == SHA_GOLD and not dif_rec))
else:
    chequeo("benchmark_kmeans.json", False, "no está", omitido=True)

## 8. Resumen de la auditoría

In [ ]:
resumen = pd.DataFrame(CHEQUEOS)
display(resumen)
n_dif = int((resumen["estado"] == "DIFERENCIA").sum())
hoy = dt.datetime.now().strftime("%Y-%m-%d %H:%M")
L = ["# Auditoría de las salidas de Go", "",
     f"> Generado por `notebooks/03_auditoria_figuras.ipynb` el {hoy}. Bronze: `{BRONZE.name if BRONZE else '—'}`. "
     f"Receta (`configs/kmeans.json`): {', '.join(FEATS)}; log1p en {', '.join(LOGS) or 'ninguna'}; nivel {NIVEL}; k = {K}. "
     f"Config de limpieza: `configs/limpieza.json`. Tolerancias: gold {TOL_GOLD:g}, estandarización {TOL_ESTAND:g}, "
     f"perfiles {TOL_PERFIL:g}, scikit-learn {TOL_SKLEARN:g}; silver byte a byte."]
if SINTETICO:
    L += ["", "> **DATOS SINTÉTICOS** (muestra de `data/sample/` o `notebooks/sintetico.py`): prueban el sistema, no son mediciones."]
L += ["", f"**{(resumen['estado'] == 'OK').sum()} OK, {n_dif} con diferencias, {(resumen['estado'] == 'AVISO').sum()} avisos, "
      f"{(resumen['estado'] == 'OMITIDO').sum()} omitidas.** DIFERENCIA = bug en Go o en Python; AVISO = no es un bug pero hay "
      "que mirarlo; OMITIDO = falta la salida de Go.", "", al.tabla_md(resumen)]
(REPORTS / "auditoria.md").write_text("\n".join(L) + "\n", encoding="utf-8")
(REPORTS / "auditoria.json").write_text(json.dumps({"fecha": hoy, "bronze": BRONZE.name if BRONZE else None, "receta": receta, "chequeos": CHEQUEOS},
                                                   indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("Escrito", REPORTS / "auditoria.md")
if n_dif:
    print(f"AVISO: {n_dif} comprobaciones con diferencias: hay un bug en Go o en Python (ver el detalle).")

## 9. Figuras del informe

Con los nombres que esperan las secciones del informe. Los datos salen de las salidas de Go
(`regimenes.json`, `kmeans.json`, `codo.json`, `benchmark_*.json`, silver y asignaciones). Si Go
todavía no corrió, los regímenes salen de scikit-learn y la figura lo dice. Colores fijos por régimen
(por su número, no por su rango); los nombres «bajo», «medio» y «alto» salen de su PM medio. Cada figura
se muestra aquí y se guarda.

In [ ]:
if asig_go is not None and Cz_go is not None and GO["regimenes"].exists():
    FUENTE_REG = "Go"
    asig, Cz, est_fig = asig_go, Cz_go, est_go
    regs = leer_json(GO["regimenes"])["regimenes"]
else:
    FUENTE_REG = "scikit-learn (Go todavía no corrió)"
    km_tmp = KMeans(n_clusters=K, init="k-means++", n_init=10, random_state=int(receta.get("semilla", 42))).fit(gold_py[FEATS].to_numpy(float))
    asig = gold_py[["estacion", "fecha_hora"]].assign(regimen=km_tmp.labels_)
    Cz, est_fig = km_tmp.cluster_centers_, stats_py
    regs = al.perfilar(gold_py, km_tmp.labels_, K, stats_py)
print("Regímenes de:", FUENTE_REG)
silver_fig = silver_go if silver_go is not None else ancho_py
KR = len(regs)
ECA_TXT = {"excede": " · excede ECA", "no_evaluable": " · ECA no evaluable"}
orden_r = [r["regimen"] for r in sorted(regs, key=lambda r: -(r.get("pm10_medio") or r.get("pm2_5_medio") or 0))]
# Nombre por nivel: el régimen de menor PM medio es «bajo», el de mayor, «alto» (solo hasta k = 4).
NIVELES_TXT = {2: ["bajo", "alto"], 3: ["bajo", "medio", "alto"], 4: ["bajo", "medio", "alto", "muy alto"]}
NOMBRE_REG = dict(zip(orden_r[::-1], NIVELES_TXT.get(KR, [""] * KR)))
ETIQ = {r["regimen"]: f"R{r['regimen']} {NOMBRE_REG[r['regimen']]}".strip() + ECA_TXT.get(r.get("eca"), "") for r in regs}
R_ALTO = orden_r[0]
# La hora de fecha_hora: con «fin», la 01 es el promedio de 00:00 a 01:00 (convención de OEFA).
CONV_HORA = (leer_json(GO["regimenes"]).get("convencion_hora") if FUENTE_REG == "Go" else None) or \
    (cfg.get("hora_por_posicion") or {}).get("convencion")
NOTA_FUENTE = (f"Regímenes: {FUENTE_REG}. Receta: {', '.join(al.NOMBRE_TXT.get(f, f) for f in FEATS)}; "
               f"log1p en {', '.join(al.NOMBRE_TXT.get(f, f) for f in LOGS) or 'ninguna'}; "
               f"nivel {NIVEL}; k = {KR}.")
MESES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "set", "oct", "nov", "dic"]
display(pd.DataFrame([{"nombre": ETIQ[r["regimen"]], **{k: r.get(k) for k in ["regimen", "tamano", "fraccion", "pm2_5_medio", "pm10_medio", "eca"]}}
                      for r in regs]).round(3))

In [ ]:
# perfiles.png: media estandarizada de cada feature (el centroide) con su media en unidades originales
# (perfil de regimenes.json), y el PM medio de cada régimen frente al ECA.
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 0.55 * KR + 2.4), gridspec_kw={"width_ratios": [1.25, 1]})
Z = Cz[orden_r]
lim = max(1.0, min(3.0, float(np.abs(Z).max())))
im = a1.imshow(Z, cmap="aire_div", vmin=-lim, vmax=lim, aspect="auto")
a1.set_xticks(range(len(FEATS)), [f"{al.NOMBRE_TXT.get(f, f)}\n({al.UNIDAD_TXT.get(f, '')})" for f in FEATS])
a1.set_yticks(range(KR), [ETIQ[r] for r in orden_r])
a1.grid(False)
por_id = {r["regimen"]: r for r in regs}
for i, r in enumerate(orden_r):
    for jf, f in enumerate(FEATS):
        val = (por_id[r].get("perfil") or {}).get(f, np.nan)
        txt = "—" if val is None or not np.isfinite(val) else (f"{val:.2f}" if f == "fraccion_fina" else f"{val:.0f}" if abs(val) >= 10 else f"{val:.1f}")
        a1.text(jf, i, txt, ha="center", va="center", fontsize=9, color="white" if abs(Z[i, jf]) > 0.6 * lim else al.TINTA)
fig.colorbar(im, ax=a1, shrink=0.8, label="media estandarizada (z) = centroide")
a1.set_title("Perfil de cada régimen", loc="left", pad=26)
y = np.arange(KR)
for jp, (p, color) in enumerate([("pm10", "#25313C"), ("pm2_5", "#8c8b86")]):
    vals = [por_id[r].get(f"{p}_medio") for r in orden_r]
    if any(v is not None for v in vals):
        a2.barh(y + (0.2 if jp == 0 else -0.2), [v if v is not None else 0 for v in vals], height=0.38, color=color,
                label=f"{al.NOMBRE_TXT[p]} medio")
        a2.axvline(al.ECA[p], color=al.CRITICO, linewidth=1.1, linestyle="--")
        a2.text(al.ECA[p], -0.62, f"ECA {al.NOMBRE_TXT[p]}\n24 h = {al.ECA[p]:.0f}", color=al.CRITICO, fontsize=7.5,
                ha="center", va="bottom", clip_on=False)
a2.set_yticks(y, [f"{ETIQ[r]} · {100 * por_id[r]['fraccion']:.1f} % de las horas" for r in orden_r], fontsize=8)
a2.set_ylim(KR - 0.4, -0.6)
a2.set_xlabel("µg/m³ (media aritmética de las horas del régimen)")
a2.set_title("Material particulado frente al ECA", loc="left", pad=26)
a2.grid(axis="y", visible=False)
a2.legend(loc="lower right")
fig.suptitle("Perfil de los regímenes de calidad del aire", x=0.01, ha="left", fontweight="bold", fontsize=13)
fig.tight_layout()
al.nota_al_pie(fig, NOTA_FUENTE + " Números de la izquierda: media aritmética en unidades originales (perfil de "
               "regimenes.json)" + ("; PM2.5 medio = fracción fina × PM10 lectura por lectura." if "pm2_5" not in FEATS else ".")
               + " El ECA es un promedio de 24 h: la media horaria de un régimen sobre la línea dice que el régimen es peligroso, "
               "no que haya un incumplimiento.")
guardar(fig, "perfiles.png", f"regimenes.json y kmeans.json ({FUENTE_REG})")

In [ ]:
# niveles.png: el modelo de un vistazo. Cada hora del gold en PM2.5 frente a PM10 (µg/m³, escala log),
# coloreada por su régimen; de fondo, la partición de K-Means (el centroide más cercano en el espacio
# estandarizado, la misma regla con la que Go asigna); las líneas del ECA y los centroides en µg/m³.
pts = asig.merge(silver_fig[["estacion", "fecha_hora", "pm2_5", "pm10"]], on=["estacion", "fecha_hora"], how="left")
PISO = 0.5  # µg/m³: en escala log, las lecturas en 0 se dibujan aquí
n_cero = int(((pts["pm2_5"] <= 0) | (pts["pm10"] <= 0)).sum())
lo, hi = PISO, float(np.nanmax(pts["pm10"].to_numpy(float))) * 1.4
hi_y = float(np.nanmax(pts["pm2_5"].to_numpy(float))) * 2.5
fig, ax = plt.subplots(figsize=(9.5, 7))
ax.set_xscale("log")
ax.set_yscale("log")
if NIVEL == "absoluto" and set(FEATS) <= {"pm2_5", "pm10", "fraccion_fina"}:
    from matplotlib.colors import ListedColormap
    GX, GY = np.meshgrid(np.geomspace(lo, hi, 600), np.geomspace(lo, hi, 600))  # PM10 en x, PM2.5 en y
    Mf = al.matriz_features(pd.DataFrame({"pm10": GX.ravel(), "pm2_5": GY.ravel()}), FEATS, LOGS).to_numpy(float)
    mu, sd = np.asarray(est_fig["media"], float), np.asarray(est_fig["desv"], float)
    Zm = np.where(sd > 0, (Mf - mu) / np.where(sd > 0, sd, 1.0), 0.0)
    ax.pcolormesh(GX, GY, al.asignar(np.nan_to_num(Zm), Cz)[0].reshape(GX.shape), shading="auto", rasterized=True,
                  cmap=ListedColormap([al.color_regimen(i) for i in range(KR)]), vmin=-0.5, vmax=KR - 0.5, alpha=0.10, linewidth=0)
for r in sorted(por_id, key=lambda r: -por_id[r]["tamano"]):  # el más grande abajo, el más raro encima
    p = pts[pts["regimen"] == r]
    ax.scatter(p["pm10"].clip(lower=PISO), p["pm2_5"].clip(lower=PISO), s=4, alpha=0.35, color=al.color_regimen(r),
               linewidths=0, rasterized=True, label=f"{ETIQ[r]} · {100 * por_id[r]['fraccion']:.1f} % de las horas")
ax.plot([lo, hi], [lo, hi], color=al.TINTA_2, linewidth=0.8, linestyle=":")
ax.annotate("PM2.5 = PM10", (hi_y / 1.5, hi_y / 1.5), xytext=(-6, 0), textcoords="offset points", color=al.TINTA_2,
            fontsize=8, ha="right", va="center")
ax.axvline(al.ECA["pm10"], color=al.CRITICO, linewidth=1.1, linestyle="--")
ax.axhline(al.ECA["pm2_5"], color=al.CRITICO, linewidth=1.1, linestyle="--")
ax.text(al.ECA["pm10"] * 1.06, lo * 1.15, "ECA PM10 (24 h) = 100", color=al.CRITICO, fontsize=8, rotation=90, va="bottom")
ax.text(lo * 1.15, al.ECA["pm2_5"] * 1.06, "ECA PM2.5 (24 h) = 50", color=al.CRITICO, fontsize=8, va="bottom")
Co = al.desestandarizar(Cz, est_fig)  # centroides en unidades originales, en el orden de FEATS (nivel absoluto)
if Co is not None and "pm10" in FEATS:
    for r in range(KR):
        c = dict(zip(FEATS, Co[r]))
        x, y = c["pm10"], c.get("pm2_5", c.get("fraccion_fina", np.nan) * c["pm10"])
        ax.scatter([x], [y], s=260, color=al.SUPERFICIE, linewidth=0, zorder=5)  # anillo claro alrededor
        ax.scatter([x], [y], s=150, color=al.color_regimen(r), edgecolor=al.TINTA, linewidth=1.6, zorder=6)
        ax.annotate(f"centroide R{r} {NOMBRE_REG.get(r, '')}\nPM2.5 {y:.0f} · PM10 {x:.0f} µg/m³", (x, y), xytext=(11, -2),
                    textcoords="offset points", fontsize=8.5, color=al.TINTA, va="top", zorder=6,
                    bbox=dict(boxstyle="round,pad=0.25", fc=al.SUPERFICIE, ec="none", alpha=0.85))
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi_y)
ax.set_xlabel("PM10 (µg/m³, escala logarítmica)")
ax.set_ylabel("PM2.5 (µg/m³, escala logarítmica)")
from matplotlib.lines import Line2D
manijas = [Line2D([], [], marker="o", linestyle="", markersize=7, color=al.color_regimen(r), alpha=0.8,
                  label=f"{ETIQ[r]} · {100 * por_id[r]['fraccion']:.1f} % de las horas") for r in orden_r[::-1]]
manijas.append(Line2D([], [], marker="o", linestyle="", markersize=9, markerfacecolor=al.SUPERFICIE,
                      markeredgecolor=al.TINTA, markeredgewidth=1.6, label="centroide (media de las horas del régimen)"))
ax.legend(handles=manijas, loc="upper left", title="Régimen (puntos y fondo)", fontsize=8.5)
ax.set_title(f"{KR} niveles de aire: PM2.5 frente a PM10 por régimen" if KR in NIVELES_TXT else
             "Regímenes de K-Means: PM2.5 frente a PM10", loc="left")
al.nota_al_pie(fig, NOTA_FUENTE + f" Cada punto es una hora del gold ({al.miles(len(pts))}). Fondo: régimen del centroide más\n"
               "cercano en el espacio estandarizado (la regla de asignación de Go): en escala log, las fronteras rectas se ven curvas. "
               f"Las {al.miles(n_cero)} horas\ncon PM en 0 se dibujan en {PISO} µg/m³. El ECA es de 24 h: una hora sobre la línea no es "
               "por sí sola un incumplimiento.")
guardar(fig, "niveles.png", f"asignaciones, silver, kmeans.json y estandarizacion.json ({FUENTE_REG})")

In [ ]:
# regimen-hora.png: % de las horas de cada régimen que caen en cada hora del día (por_hora).
H = np.array([por_id[r]["por_hora"] for r in orden_r], dtype=float)
Hp = 100 * H / np.maximum(H.sum(axis=1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(11, 0.45 * KR + 1.8))
im = ax.imshow(Hp, cmap="aire_seq", aspect="auto", vmin=0)
ax.set_xticks(range(24), [f"{h:02d}" for h in range(24)], fontsize=8)
ax.set_yticks(range(KR), [ETIQ[r] for r in orden_r])
ax.set_xlabel("Hora del día en Lima" + (" (fin del promedio horario: la 01 es el promedio de 00:00 a 01:00)"
                                         if CONV_HORA == "fin" else ""))
ax.grid(False)
fig.colorbar(im, ax=ax, label="% de las horas del régimen", shrink=0.85)
ax.set_title("¿A qué hora aparece cada régimen?", loc="left")
al.nota_al_pie(fig, NOTA_FUENTE + " Cada fila suma 100 %; una fila plana es un régimen sin hora preferida.")
guardar(fig, "regimen-hora.png", f"regimenes.json, por_hora ({FUENTE_REG})")

# regimen-estacion.png: % de las horas de cada estación en cada régimen (por_estacion).
ests = sorted({e for r in regs for e in (r.get("por_estacion") or {})})
E = pd.DataFrame({r: [(por_id[r].get("por_estacion") or {}).get(e, 0) for e in ests] for r in orden_r}, index=ests, dtype=float)
Ep = 100 * E.div(E.sum(axis=1).replace(0, np.nan), axis=0)
ubic = (lim_go or rep_py or {}).get("estaciones", {})
dep = {e: al.NOMBRE_REGION.get(al.region((ubic.get(e, {}).get("ubicacion") or {}).get("departamento")), "") for e in ests}
Ep = Ep.loc[sorted(Ep.index, key=lambda e: (dep.get(e, ""), e))]
fig, ax = plt.subplots(figsize=(1.0 * KR + 4.5, 0.32 * len(Ep) + 1.8))
im = ax.imshow(Ep.to_numpy(), cmap="aire_seq", aspect="auto", vmin=0, vmax=100)
ax.set_xticks(range(KR), [ETIQ[r].replace(" · ", "\n") for r in orden_r], fontsize=8)
ax.set_yticks(range(len(Ep)), [f"{e} ({dep[e]})" if dep.get(e) else e for e in Ep.index], fontsize=8)
ax.grid(False)
if len(Ep) <= 40:
    for i in range(len(Ep)):
        for jj in range(KR):
            v = Ep.iat[i, jj]
            ax.text(jj, i, f"{v:.0f}", ha="center", va="center", fontsize=7, color="white" if v > 55 else al.TINTA)
fig.colorbar(im, ax=ax, label="% de las horas de la estación", shrink=0.85)
ax.set_title("¿Qué régimen domina en cada estación?", loc="left")
al.nota_al_pie(fig, NOTA_FUENTE + " Cada fila suma 100 %.")
guardar(fig, "regimen-estacion.png", f"regimenes.json, por_estacion ({FUENTE_REG})")

In [ ]:
# episodio.png (la estrella): PM2.5 y PM10 de una estación durante unos días, con el fondo coloreado
# por régimen, las líneas del ECA y una franja con las horas cuyo promedio de 24 h supera el ECA.
# Estación: la que tiene más horas en el régimen más alto (regimenes.json, por_estacion). Ventana: la de
# EPISODIO_DIAS días con más horas de 24 h sobre el ECA más horas en el régimen alto, entre las que tienen
# régimen en >= 60 % de las horas (si no hay horas sobre el ECA, la de PM10 medio más alto).
EPISODIO_DIAS = 7
ESTACION_EPISODIO = os.environ.get("AIRE_EPISODIO")  # None: la estación con más horas en el régimen más alto
horas_alto = pd.Series(por_id[R_ALTO].get("por_estacion") or {}, dtype=float).sort_values(ascending=False)
display(horas_alto.head(5).astype(int).rename(f"horas en {ETIQ[R_ALTO]}").to_frame())
if not ESTACION_EPISODIO and len(horas_alto):
    ESTACION_EPISODIO = str(horas_alto.index[0])

rej = al.regularizar(silver_fig[silver_fig["estacion"] == ESTACION_EPISODIO] if ESTACION_EPISODIO else silver_fig,
                     [c for c in ["pm2_5", "pm10"] if c in silver_fig])
mv = al.movil_24h(rej)
rej = rej.merge(mv[["estacion", "fecha_hora", "excede_24h"]], on=["estacion", "fecha_hora"]).merge(asig, on=["estacion", "fecha_hora"], how="left")
rej["dia"] = rej["fecha_hora"].dt.normalize()
rej["alto"] = rej["regimen"] == R_ALTO
diario = rej.groupby(["estacion", "dia"]).agg(exc=("excede_24h", "sum"), alto=("alto", "sum"), asignadas=("regimen", "count"),
                                              pm=("pm10" if "pm10" in rej else "pm2_5", "mean"))
mejor = None
for est, g in diario.groupby(level=0):
    g = g.droplevel(0).asfreq("D")
    n_dias = min(EPISODIO_DIAS, len(g))
    cobertura = g["asignadas"].fillna(0).rolling(n_dias).sum() / (24 * n_dias)
    puntaje = ((g["exc"].fillna(0) + g["alto"].fillna(0)).rolling(n_dias).sum() * 1e6
               + g["pm"].rolling(n_dias, min_periods=1).mean()).where(cobertura >= 0.6)
    if puntaje.notna().any() and (mejor is None or puntaje.max() > mejor[2]):
        mejor = (est, puntaje.idxmax(), float(puntaje.max()), n_dias)
if mejor is None:
    print("No hay una ventana con regímenes asignados suficientes para la figura del episodio.")
else:
    est, fin_dia, punt, n_dias = mejor
    ini = fin_dia - pd.Timedelta(days=n_dias - 1)
    w = rej[(rej["estacion"] == est) & (rej["dia"] >= ini) & (rej["dia"] <= fin_dia)].reset_index(drop=True)
    fig, (ax, axr, axe) = plt.subplots(3, 1, figsize=(12, 5.6), sharex=True,
                                       gridspec_kw={"height_ratios": [6, 0.5, 0.5], "hspace": 0.1})
    t = w["fecha_hora"]
    paso = pd.Timedelta(hours=1)
    reg_w = w["regimen"]
    for _, tramo in w.groupby((reg_w != reg_w.shift()).cumsum()):
        r = tramo["regimen"].iloc[0]
        if pd.isna(r):
            continue
        x0, x1 = tramo["fecha_hora"].iloc[0], tramo["fecha_hora"].iloc[-1] + paso
        ax.axvspan(x0, x1, color=al.color_regimen(int(r)), alpha=0.16, linewidth=0)
        axr.axvspan(x0, x1, color=al.color_regimen(int(r)), alpha=0.9, linewidth=0)
    exc_w = w["excede_24h"].astype(bool)
    for _, tramo in w[exc_w].groupby((exc_w != exc_w.shift()).cumsum()[exc_w]):
        axe.axvspan(tramo["fecha_hora"].iloc[0], tramo["fecha_hora"].iloc[-1] + paso, color=al.TINTA_2, alpha=0.85, linewidth=0)
    from matplotlib.lines import Line2D
    from matplotlib.transforms import blended_transform_factory
    a_la_derecha = blended_transform_factory(ax.transAxes, ax.transData)
    series = []
    for p, color, lw in [("pm10", "#0b0b0b", 1.6), ("pm2_5", "#52514e", 1.2)]:
        if p in w:
            series += ax.plot(t, w[p], color=color, linewidth=lw, label=f"{al.NOMBRE_TXT[p]} horario")
            ax.axhline(al.ECA[p], color=al.CRITICO, linewidth=1.1, linestyle="--")
            ax.text(1.005, al.ECA[p], f"ECA {al.NOMBRE_TXT[p]} (24 h) = {al.ECA[p]:.0f}", transform=a_la_derecha,
                    color=al.CRITICO, fontsize=8, va="center", ha="left")
    ax.set_ylabel("µg/m³")
    ax.set_ylim(bottom=0)
    ax.grid(axis="x", visible=False)
    presentes = [int(r) for r in pd.unique(reg_w.dropna())]
    ax.legend(handles=series + [Line2D([], [], color=al.CRITICO, linewidth=1.1, linestyle="--", label="ECA de 24 h")]
              + [Patch(color=al.color_regimen(r), alpha=0.6, label=f"fondo: {ETIQ[r]}") for r in orden_r if r in presentes],
              loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
    for a, txt in [(axr, "régimen"), (axe, "24 h > ECA")]:
        a.set_yticks([])
        a.set_ylabel(txt, rotation=0, ha="right", va="center", fontsize=8)
        a.grid(False)
    axe.xaxis.set_major_locator(mdates.DayLocator())
    axe.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: (lambda d: f"{d.day} {MESES[d.month - 1]}")(mdates.num2date(x))))
    lugar = (dep or {}).get(est) if "dep" in globals() else None
    ax.set_title(f"Un episodio en {est}{f' ({lugar})' if lugar else ''}: {ini.day} {MESES[ini.month - 1]} – {fin_dia.day} {MESES[fin_dia.month - 1]} {fin_dia.year}",
                 loc="left", fontsize=13)
    EPISODIO = {"estacion": est, "desde": f"{ini:%Y-%m-%d}", "hasta": f"{fin_dia:%Y-%m-%d}", "dias": n_dias,
                "horas_alto_en_la_ventana": int(w["alto"].sum()), "horas_24h_sobre_eca_en_la_ventana": int(exc_w.sum()),
                "horas_alto_de_la_estacion": int(horas_alto.get(est, 0))}
    al.nota_al_pie(fig, NOTA_FUENTE + f" Estación con más horas en {ETIQ[R_ALTO]} ({EPISODIO['horas_alto_de_la_estacion']}); ventana de "
                   f"{n_dias} días con más horas en ese régimen y con el promedio de 24 h sobre el ECA.\nEl ECA es un promedio de 24 h: "
                   "una hora sobre la línea no es por sí sola un incumplimiento; la franja «24 h > ECA» marca las horas cuyo promedio "
                   "de las 24 h que terminan en ella lo supera.\nSin fondo: horas sin régimen (falta alguna feature).")
    guardar(fig, "episodio.png", f"silver y asignaciones ({FUENTE_REG}), {est}, {ini:%Y-%m-%d} a {fin_dia:%Y-%m-%d}")
    print("Episodio:", EPISODIO)

In [ ]:
# codo.png: la inercia por k de codo.json (Go) y, al lado, la silueta de scikit-learn (Go no la calcula
# hasta la PC4). Si no hay codo.json, la inercia también sale de scikit-learn.
Xc = gold_ref[FEATS].to_numpy(float)
if codo_go is not None:
    ks = [p["k"] for p in codo_go["puntos"]]
    inercias = [p["inercia"] for p in codo_go["puntos"]]
    n_codo = codo_go["puntos"][0]["n"]
    fuente_codo = "codo.json (Go)"
else:
    ks, inercias, n_codo = list(range(2, 11)), [], min(len(Xc), 200_000)
    fuente_codo = "scikit-learn"
paso_c = max(1, math.ceil(len(Xc) / n_codo))
Xs = Xc[::paso_c]
siluetas = []
for kk in ks:
    m = KMeans(n_clusters=kk, init="k-means++", n_init=3, random_state=int(receta.get("semilla", 42))).fit(Xs)
    if codo_go is None:
        inercias.append(m.inertia_)
    siluetas.append(silhouette_score(Xs, m.labels_, sample_size=min(10_000, len(Xs)), random_state=0))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
a1.plot(ks, inercias, marker="o", color=al.PALETA[0])
a1.set_title(f"Inercia ({'Go' if codo_go is not None else 'scikit-learn'})", loc="left")
a1.set_ylabel("suma de distancias al cuadrado")
a2.plot(ks, siluetas, marker="o", color=al.PALETA[0])
a2.set_title("Silueta media (scikit-learn, Colab)", loc="left")
for a in (a1, a2):
    a.axvline(K, color=al.CRITICO, linewidth=1, linestyle="--")
    a.set_xlabel("k (número de regímenes)")
    a.set_xticks(ks)
if K in ks:
    a1.annotate(f"k = {K} (receta)", (K, inercias[ks.index(K)]), xytext=(8, 8), textcoords="offset points", color=al.CRITICO, fontsize=9)
fig.suptitle("Método del codo", x=0.01, ha="left", fontweight="bold", fontsize=13)
fig.tight_layout()
al.nota_al_pie(fig, f"Inercia de {fuente_codo} sobre {al.miles(n_codo)} filas del gold (submuestra sistemática); silueta sobre "
               f"{al.miles(min(10_000, len(Xs)))} puntos. k = {K} es el de configs/kmeans.json.")
guardar(fig, "codo.png", fuente_codo)

In [ ]:
# speedup.png: speedup frente a W del cargador (benchmark_cargador.json) y del K-Means (benchmark_kmeans.json),
# con la diagonal ideal y las barras del IC bootstrap 95 % (K-Means) o del rango de las repeticiones (cargador).
paneles = [p for p in [("cargador", bc), ("kmeans", bk)] if p[1] is not None]
if not paneles:
    print("No hay benchmark_cargador.json ni benchmark_kmeans.json: no se dibuja speedup.png.")
else:
    fig, axs = plt.subplots(1, len(paneles), figsize=(5.6 * len(paneles), 4.0), squeeze=False)
    wmax = 1
    for ax, (nombre, b) in zip(axs[0], paneles):
        if nombre == "cargador":
            ms = sorted(b["mediciones"], key=lambda m: m["workers"])
            base = next(m for m in ms if m["workers"] == b.get("base_workers", 1))["mediana_s"]
            W = np.array([m["workers"] for m in ms])
            s = np.array([m["speedup"] for m in ms])
            def ic(m):
                v = m.get("speedup_ic")
                if isinstance(v, dict):
                    return v.get("inferior", m["speedup"]), v.get("superior", m["speedup"]), True
                if isinstance(v, (list, tuple)) and len(v) == 2:
                    return v[0], v[1], True
                return base / max(m["tiempos_s"]), base / min(m["tiempos_s"]), False
            lohi = [ic(m) for m in ms]
            lo, hi = np.array([x[0] for x in lohi]), np.array([x[1] for x in lohi])
            con_ic = all(x[2] for x in lohi)
            ax.errorbar(W, s, yerr=[np.maximum(s - lo, 0), np.maximum(hi - s, 0)], marker="o", color=al.PALETA[0],
                        capsize=3, label="speedup (barras: IC bootstrap 95 %)" if con_ic else "mediana (barras: rango de las repeticiones)")
            ax.set_title(f"Cargador ({al.miles(b.get('filas_largas', 0))} filas largas)", loc="left")
            wmax = max(wmax, W.max())
        else:
            res = [r for r in b.get("resumen", []) if r.get("modo") == "conc" and r.get("speedup")]
            series = sorted({(r["experimento"], r.get("chunk")) for r in res})
            for i, (exp, ch) in enumerate(series):
                rr = sorted([r for r in res if r["experimento"] == exp and r.get("chunk") == ch], key=lambda r: r["workers"])
                W = np.array([r["workers"] for r in rr])
                s = np.array([r["speedup"]["punto"] for r in rr])
                lo = np.array([r["speedup"]["inferior"] for r in rr])
                hi = np.array([r["speedup"]["superior"] for r in rr])
                ax.errorbar(W, s, yerr=[s - lo, hi - s], marker="o", color=al.PALETA[i % len(al.PALETA)], capsize=3,
                            label=f"{exp}, chunk {ch}")
                wmax = max(wmax, W.max())
            n_k = (b.get("datos") or {}).get("n")
            ax.set_title(f"K-Means ({al.miles(n_k) if n_k else '?'} filas del gold)", loc="left")
        ax.set_xlabel("workers (W)")
        ax.set_ylabel("speedup frente al secuencial")
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    for ax in axs[0]:
        ax.plot([1, wmax], [1, wmax], color=al.TINTA_2, linewidth=0.8, linestyle=":", label="ideal (lineal)")
        ax.axhline(1, color=al.REJILLA, linewidth=1)
        ax.legend(fontsize=8, loc="upper left")
    meta = (bk or {}).get("metadatos") or {}
    fig.suptitle("Speedup frente al número de workers", x=0.01, ha="left", fontweight="bold", fontsize=13)
    fig.tight_layout()
    al.nota_al_pie(fig, f"Máquina {meta.get('maquina', bc.get('host') if bc else '?')}, {meta.get('cpus_logicas', bc.get('cpus') if bc else '?')} CPU. "
                   "Barras: IC bootstrap 95 % del cociente (cargador: el de benchmark_cargador.json si lo trae; si no, el rango de las repeticiones)."
                   + NOTA_BENCH_CARGADOR)
    guardar(fig, "speedup.png", "benchmark_cargador.json y benchmark_kmeans.json")

In [ ]:
# cobertura.png: estaciones y horas con PM por región (las 25 regiones, también las que no tienen).
ubic = (lim_go or rep_py or {}).get("estaciones", {})
if ubic:
    d_est = pd.DataFrame([{"estacion": e, "region": al.region((r.get("ubicacion") or {}).get("departamento")) or "SIN DEPARTAMENTO"}
                          for e, r in ubic.items() if r.get("anchas", 0) > 0])
    hpm = silver_fig[[c for c in ["pm2_5", "pm10"] if c in silver_fig]].notna().any(axis=1).groupby(silver_fig["estacion"]).sum()
    d_est["horas_pm"] = d_est["estacion"].map(hpm).fillna(0)
    por_reg = d_est.groupby("region").agg(estaciones=("estacion", "size"), horas_pm=("horas_pm", "sum"))
    regiones = sorted(set(al.REGIONES_PERU) | set(por_reg.index.dropna()))
    por_reg = por_reg.reindex(regiones, fill_value=0)
    por_reg = por_reg.assign(_n=por_reg.index).sort_values(["estaciones", "horas_pm", "_n"], ascending=[False, False, True]).drop(columns="_n")
    con = int((por_reg.loc[por_reg.index.isin(al.REGIONES_PERU), "estaciones"] > 0).sum())
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 0.22 * len(por_reg) + 1.6), sharey=True, gridspec_kw={"wspace": 0.08})
    yy = np.arange(len(por_reg))
    colores = [al.PALETA[0] if n > 0 else "#d9d8d3" for n in por_reg["estaciones"]]
    a1.barh(yy, por_reg["estaciones"], color=colores, height=0.65)
    for yi, nn in zip(yy, por_reg["estaciones"]):
        a1.text(nn + 0.05, yi, f"{int(nn)}", va="center", fontsize=8, color=al.TINTA if nn else "#8c8b86")
    a1.set_yticks(yy, [al.NOMBRE_REGION.get(r, str(r).title()) for r in por_reg.index], fontsize=8)
    for lab, nn in zip(a1.get_yticklabels(), por_reg["estaciones"]):
        lab.set_color(al.TINTA if nn else "#8c8b86")
    a1.set_xlabel("estaciones con datos válidos")
    a1.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    a1.invert_yaxis()
    a1.set_title("Estaciones por región", loc="left")
    a2.barh(yy, por_reg["horas_pm"] / 1000, color=colores, height=0.65)
    a2.set_xlabel("miles de horas con PM2.5 o PM10 válido")
    a2.set_title("Horas con dato de material particulado", loc="left")
    for a in (a1, a2):
        a.grid(axis="y", visible=False)
    fig.suptitle(f"Cobertura de OEFA: {int(por_reg['estaciones'].sum())} estaciones en {con} de las {len(al.REGIONES_PERU)} regiones",
                 x=0.01, ha="left", fontweight="bold", fontsize=13)
    al.nota_al_pie(fig, "Una región sin estación no es una región con aire limpio: es una región sin medición.", 8)
    guardar(fig, "cobertura.png", "limpieza.json (estaciones) y silver")
print("Figuras:", sorted(p.name for p in FIGURAS.glob("*.png")))